In [1]:
# import libraries 

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Dataset Crime
crime_df = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime\Crimes_-_2001_to_Present.csv")

print(len(crime_df))
print("Total Columns:", crime_df.shape[1])


8498314
Total Columns: 22


In [3]:
for col in crime_df.columns:
    print(col)

ID
Case Number
Date
Block
IUCR
Primary Type
Description
Location Description
Arrest
Domestic
Beat
District
Ward
Community Area
FBI Code
X Coordinate
Y Coordinate
Year
Updated On
Latitude
Longitude
Location


In [4]:
crime_df.dtypes

ID                        int64
Case Number              object
Date                     object
Block                    object
IUCR                     object
Primary Type             object
Description              object
Location Description     object
Arrest                     bool
Domestic                   bool
Beat                      int64
District                float64
Ward                    float64
Community Area          float64
FBI Code                 object
X Coordinate            float64
Y Coordinate            float64
Year                      int64
Updated On               object
Latitude                float64
Longitude               float64
Location                 object
dtype: object

In [5]:
# Missing Values

missing = crime_df.isnull().sum()
missing_pct = (crime_df.isnull().sum() / len(crime_df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

                      Missing Count  Missing %
Location Description          15607       0.18
District                         47       0.00
Ward                         614818       7.23
Community Area               613685       7.22
X Coordinate                  94699       1.11
Y Coordinate                  94699       1.11
Latitude                      94699       1.11
Longitude                     94699       1.11
Location                      94699       1.11


In [6]:
print(crime_df['Year'].min())
print(crime_df['Year'].max())
print(crime_df['Year'].unique())



2001
2026
[2022 2023 2020 2017 2019 2021 2024 2008 2015 2014 2013 2018 2004 2011
 2010 2002 2006 2007 2005 2001 2016 2012 2009 2003 2025 2026]


# Data Cleaning

In [7]:
print(len(crime_df))

8498314


In [8]:
# Filter years 2001-2023

crime_df = crime_df[(crime_df['Year']>=2001) & (crime_df['Year']<=2023)].copy()
len(crime_df)

7980470

In [9]:
# Convert Data column to date and time

crime_df['Date'] = pd.to_datetime(crime_df['Date'],format='%m/%d/%Y %I:%M:%S %p')

crime_df['month']       = crime_df['Date'].dt.month
crime_df['day_of_week'] = crime_df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
crime_df['hour']        = crime_df['Date'].dt.hour
crime_df['is_weekend']  = crime_df['day_of_week'].isin([5, 6]).astype(int)

In [10]:
# Missing Community Area

missing_community = crime_df['Community Area'].isnull().sum()
int(missing_community)



613682

In [ ]:
# Fill Missing community area using latitude and longitude + Boundary CSV

import geopandas as gpd
from shapely.geometry import Point

community = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime\Boundaries_-_Community_Areas_20260429.csv")
print(community.columns)




Index(['the_geom', 'AREA_NUMBE', 'COMMUNITY', 'AREA_NUM_1', 'SHAPE_AREA',
       'SHAPE_LEN'],
      dtype='object')


In [12]:
# Convert boundry csv to GeoDataFrame

community['geometry'] = gpd.GeoSeries.from_wkt(community['the_geom'])
community_gdf = gpd.GeoDataFrame(community, geometry='geometry', crs="EPSG:4326")

missing_df = crime_df[crime_df["Community Area"].isna()].copy()

# Create geometry from latitude and longitude
geometry = [Point(xy) for xy in zip(missing_df["Longitude"], missing_df["Latitude"])]

missing_gdf = gpd.GeoDataFrame(missing_df, geometry=geometry, crs="EPSG:4326")

# Spatial Join
joined = gpd.sjoin(missing_gdf,community_gdf, how="left", predicate="within")

# Filling into original Dataset
crime_df.loc[joined.index, "Community Area"] = joined["AREA_NUM_1"]

# Missing values
print("Remaining missing values:", crime_df["Community Area"].isna().sum())


Remaining missing values: 9736


In [13]:
len(crime_df)


7980470

In [14]:
crime_df = crime_df.dropna(subset=['Community Area'])
len(crime_df)

7970734

In [15]:
# Convert Community to int
crime_df['Community Area'] = crime_df['Community Area'].astype(int)

# COnvert Arrest to int 
crime_df['Arrest']   = crime_df['Arrest'].astype(int)
crime_df['Domestic'] = crime_df['Domestic'].astype(int)

In [16]:
# Drop the unnecessary cols
drop_cols = ['Case Number', 'Block','Ward', 'IUCR', 'X Coordinate', 'Y Coordinate', 'Updated On', 'Location']
crime_df = crime_df.drop(columns=drop_cols)

In [17]:
print(len(crime_df))
print("Total Cols:", crime_df.shape[1])
print("Colunm Names", crime_df.columns.tolist())
print("Arrest = 1:", crime_df['Arrest'].sum())
print("Arrest = 0:", (crime_df['Arrest']==0).sum())
print("Arrest Rate:", crime_df['Arrest'].mean()*100)

print("Remaining missing values:\n", crime_df.isnull().sum()[crime_df.isnull().sum()>0])



7970734
Total Cols: 18
Colunm Names ['ID', 'Date', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Community Area', 'FBI Code', 'Year', 'Latitude', 'Longitude', 'month', 'day_of_week', 'hour', 'is_weekend']
Arrest = 1: 2056813
Arrest = 0: 5913921
Arrest Rate: 25.804562039079464
Remaining missing values:
 Location Description    13301
District                   47
Latitude                83891
Longitude               83891
dtype: int64


In [18]:
# Fill district using Mode per beat
crime_df["District"] = crime_df["District"].fillna(crime_df.groupby("Beat")["District"].transform(lambda x: x.mode()[0]))

# Filling Missing Latitude and longitude using Beat and District
beat_coords = crime_df.groupby("Beat")[["Latitude", "Longitude"]].mean()
crime_df["Latitude"] = crime_df["Latitude"].fillna(crime_df["Beat"].map(beat_coords["Latitude"]))
crime_df["Longitude"] = crime_df["Longitude"].fillna(crime_df["Beat"].map(beat_coords["Longitude"]))

print(crime_df[["Latitude", "Longitude", "District"]].isnull().sum())

Latitude     0
Longitude    0
District     0
dtype: int64


In [19]:
# For the Location Description we can see that similar crimes happen in similar places 
# Like Theft --> Street, Burglary --> Residence
# Fill Location Description using mode per Primary Type
crime_df["Location Description"] = crime_df["Location Description"].fillna(crime_df.groupby("Primary Type")["Location Description"].transform(lambda x: x.mode()[0]))
print(crime_df[["Location Description"]].isnull().sum())



Location Description    0
dtype: int64


In [20]:
print("Remaining missing values:\n", crime_df.isnull().sum()[crime_df.isnull().sum()>0])


Remaining missing values:
 Series([], dtype: int64)


In [21]:
print(crime_df.shape)

(7970734, 18)


In [ ]:
crime_df.to_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime\Crime_2001_2023_Cleaned.csv", index=False)

In [26]:
df2 = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime\Crime_2001_2023_Cleaned.csv")

In [27]:
df2.shape

(7970734, 18)